# Inventory Image Tagging via Handcrafted Features and a Small CNN: Phase 1 (ML evaluation)
Fashion-MNIST (official split). 9 final models: {LogReg, k-NN, RF, RBF-SVM} x {pixels, HOG} + 1 CNN.
**Run top to bottom, no manual steps.** Held-out data is first loaded in section 8 and nowhere earlier.
All numbers below are produced by code in this notebook; nothing is hand-typed.

## 0. Setup and environment

In [ ]:
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src/config.py").exists():
    repo = os.environ.get("FASHION_REPO_URL", "https://github.com/manas-2212/fashion-tagging.git")  # URL as given in README.md
    subprocess.run(["git", "clone", repo, "fashion-tagging"], check=True)
    os.chdir("fashion-tagging")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import copy, json, time, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import (SEED, CLASS_NAMES, NUM_CLASSES, N_TRAIN, N_VAL, PIXEL_DIM, HOG_PARAMS, HOG_DIM,
                        BENCH_BATCH_SIZE, SVM_TUNE_N, ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, set_global_seed)
from src.data import (load_training_dataset, load_test_dataset, verify_training_dataset, verify_test_dataset,
                      prepare_splits, find_duplicates, pca_projection, class_centroid_distances, sanity_baselines)
from src.features import flatten_pixels, extract_hog_features, hog_feature_dim, plot_hog_example, cached_features
from src.classical_models import overfit_check, classical_param_count
from src.evaluation import (evaluate_predictions, top_confusions, plot_confusion_matrix, show_prediction_examples,
                            compare_errors, plot_model_comparison, plot_per_class_recall_heatmap, bootstrap_ci)
from src.experiments import (build_feature_sets, run_classical_tuning, run_classical_final_fits, freeze_selection)
from src.reporting import get_environment_info, RESULT_COLUMNS, build_results_table, save_results_table
from src.cnn import (compute_mean_std, make_cnn_tensor, build_cnn, train_cnn, predict_cnn, plot_learning_curves)
from src.benchmarking import count_trainable_params, benchmark_cnn_inference, benchmark_classical_inference

RESULTS_DIR.mkdir(exist_ok=True); FIGURES_DIR.mkdir(exist_ok=True)
set_global_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = get_environment_info()
print(json.dumps(env, indent=2, default=str))
(RESULTS_DIR / "environment.json").write_text(json.dumps(env, indent=2, default=str))
print("training device:", device, "| torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())

## 1. Load and integrity checks

In [ ]:
# Pre-gate: load and validate ONLY the official training split.
X_train_full, y_train_full = load_training_dataset()
integrity = verify_training_dataset(X_train_full, y_train_full)
print(integrity)
print(X_train_full.shape, X_train_full.dtype, y_train_full.shape, y_train_full.dtype)


## 2. EDA and data quality (Akhil)
All statistics use the official 60k train images or the 54k train split (same deterministic split as section 3). No held-out image or label is used here.
Held-out class balance (1,000 per class) is asserted inside `verify_dataset` in section 1 and is not plotted.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from src.config import CLASS_NAMES, FIGURES_DIR, RESULTS_DIR, SEED
from src.data import class_centroid_distances, pca_projection, prepare_splits, stratified_subset

FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
s = prepare_splits(X_train_full, y_train_full)   # same deterministic split as section 3
X_tr, y_tr = s["X_train"], s["y_train"]
rng = np.random.default_rng(SEED)

In [ ]:
# a) Class distribution (official 60k train)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(10), np.bincount(y_train_full, minlength=10), 0.6, label="Train (official 60k)")
ax.set_xticks(np.arange(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
ax.set_title("Fashion-MNIST class distribution (train)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_class_distribution.png", dpi=150)
plt.show()

In [ ]:
# b) 5 random images per class (fixed seed, from the train split)
fig, axes = plt.subplots(10, 5, figsize=(6, 12))
for c in range(10):
    idx = rng.choice(np.where(y_tr == c)[0], 5, replace=False)
    for j, i in enumerate(idx):
        ax = axes[c, j]
        ax.imshow(X_tr[i], cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], rotation=0, ha="right", va="center", fontsize=8)
fig.suptitle("Five random training images per class")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_samples_per_class.png", dpi=150)
plt.show()

In [ ]:
# c) Mean image per class (train split)
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(X_tr[y_tr == c].mean(axis=0), cmap="gray")
    ax.set_title(CLASS_NAMES[c], fontsize=9)
    ax.axis("off")
fig.suptitle("Mean image per class (54k train split)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_mean_images.png", dpi=150)
plt.show()

In [ ]:
# d) Confusable upper-body classes: 8 examples each
panel = [6, 0, 2, 4]  # Shirt, T-shirt/top, Pullover, Coat
fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
for r, c in enumerate(panel):
    idx = rng.choice(np.where(y_tr == c)[0], 8, replace=False)
    for j, i in enumerate(idx):
        ax = axes[r, j]
        ax.imshow(X_tr[i], cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle("Shirt vs T-shirt/top vs Pullover vs Coat (8 random training images each)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_confusable_panel.png", dpi=150)
plt.show()

In [ ]:
# e) Pixel statistics on the TRAIN split only (scale 0-1)
pix = {
    "n_images": int(len(X_tr)),
    "mean": float(X_tr.mean(dtype=np.float64) / 255),
    "std": float(X_tr.std(dtype=np.float64) / 255),
    "fraction_zero_pixels": float((X_tr == 0).mean()),
    "per_class_mean_intensity": {CLASS_NAMES[c]: float(X_tr[y_tr == c].mean(dtype=np.float64) / 255)
                                 for c in range(10)},
}
print(json.dumps(pix, indent=2))
dq_path = RESULTS_DIR / "data_quality.json"
dq = json.loads(dq_path.read_text()) if dq_path.exists() else {}
dq["pixel_stats_train54k"] = pix

In [ ]:
# f) PCA (fit on the 54k train split only) and class-centroid distances
Z, evr = pca_projection(X_tr, 50)
dq["pca_evr_first2"] = [float(v) for v in evr[:2]]
print("PCA variance, PC1 and PC2:", dq["pca_evr_first2"], "| first 10:", float(evr[:10].sum()), "| 50:", float(evr.sum()))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, 51), np.cumsum(evr), marker="o", markersize=3)
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA explained variance (fit on 54k train split)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_pca_variance.png", dpi=150)
plt.show()

Zs, ys, _ = stratified_subset(Z, y_tr, 5000)  # 5000 points for readability, fixed seed
fig, ax = plt.subplots(figsize=(8, 6))
for c in range(10):
    ax.scatter(Zs[ys == c, 0], Zs[ys == c, 1], s=4, alpha=0.6, color=plt.cm.tab10(c), label=CLASS_NAMES[c])
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("PCA 2D projection (5000 stratified train images)")
ax.legend(markerscale=4, fontsize=8, loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_pca_scatter.png", dpi=150)
plt.show()

D = class_centroid_distances(X_tr, y_tr)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D.values, cmap="viridis_r")
ax.set_xticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(10))
ax.set_yticklabels(CLASS_NAMES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{D.values[i, j]:.1f}", ha="center", va="center", fontsize=6, color="white")
ax.set_title("Euclidean distance between class mean images (train split, pixels in 0-1)")
fig.colorbar(im, ax=ax, label="Distance")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_centroid_heatmap.png", dpi=150)
plt.show()

dq_path.write_text(json.dumps(dq, indent=2))
print("saved figures:", sorted(p.name for p in FIGURES_DIR.glob("eda_*.png")))

## 3. Stratified split and audit (54,000 train / 6,000 validation, seed 42)

In [ ]:
splits = prepare_splits(X_train_full, y_train_full)
X_train, y_train, X_val, y_val = splits["X_train"], splits["y_train"], splits["X_val"], splits["y_val"]
assert len(X_train) == N_TRAIN and len(X_val) == N_VAL
assert not set(splits["idx_train"]) & set(splits["idx_val"]), "train/val index overlap"
assert np.array_equal(np.bincount(y_val, minlength=NUM_CLASSES), np.full(NUM_CLASSES, N_VAL // NUM_CLASSES)), "val not stratified"
print("train/val duplicate audit:", find_duplicates(X_train, X_val))
display(sanity_baselines(X_train, y_train, X_val, y_val))       # validation only; not part of the 9-model table
# PCA (fit on the train split only) is computed once, in section 2

## 4. Features (pixels and HOG), cached; no held-out data

In [ ]:
assert HOG_DIM == hog_feature_dim(**HOG_PARAMS), "HOG_DIM in config does not match the HOG parameters"
feats = build_feature_sets(X_train, X_val)
for rep, dim in (("pixels", PIXEL_DIM), ("hog", HOG_DIM)):
    for part, n in (("train", N_TRAIN), ("val", N_VAL)):
        a = feats[rep][part]
        assert a.shape == (n, dim) and a.dtype == np.float32, (rep, part, a.shape, a.dtype)
        assert np.isfinite(a).all()
print({rep: {p: feats[rep][p].shape for p in feats[rep]} for rep in feats})
fig = plot_hog_example(X_train, 0); fig.savefig(FIGURES_DIR / "hog_example.png", dpi=150); plt.show()

## 5. Classical tuning (validation macro-F1; SVM on a stratified 10k subset)

In [ ]:
best_params = run_classical_tuning(feats, y_train, y_val)
for k, v in best_params.items(): print(k, v)

## 6. Classical final fits (54k train only; runtime gate; overfit check)

In [ ]:
models, fit_infos = run_classical_final_fits(feats, y_train, best_params)
rows = []
for (name, rep), pipe in models.items():
    oc = overfit_check(pipe, feats[rep]["train"], y_train, feats[rep]["val"], y_val)
    rows.append({"model": name, "rep": rep, **oc})
overfit_df = pd.DataFrame(rows); display(overfit_df.round(4))

## 7. CNN training (train 54k / validation 6k only)
Train loss/accuracy are running averages in train mode (dropout on); validation is evaluated in eval mode after each epoch.
Early stopping on validation loss; the best-epoch weights are restored.

In [ ]:
CNN_CFG = dict(lr=1e-3, batch_size=128, max_epochs=30, patience=5, dropout=0.5, seed=SEED)
cnn_mean, cnn_std = compute_mean_std(X_train)                    # train split only
X_tr_t = make_cnn_tensor(X_train, cnn_mean, cnn_std)
X_val_t = make_cnn_tensor(X_val, cnn_mean, cnn_std)
assert X_tr_t.shape == (N_TRAIN, 1, 28, 28) and X_tr_t.dtype == torch.float32
print("train-split mean/std: %.5f / %.5f | tensor mean/std: %.4f / %.4f" % (cnn_mean, cnn_std, X_tr_t.mean(), X_tr_t.std()))

set_global_seed(SEED)
_probe = build_cnn(CNN_CFG["dropout"])
print("output shape for randn(8,1,28,28):", tuple(_probe(torch.randn(8, 1, 28, 28)).shape), "| trainable params:", count_trainable_params(_probe))
assert count_trainable_params(_probe) == 241546

In [ ]:
# time ONE real epoch on this device before committing to the full run (throwaway model)
set_global_seed(SEED)
_, _, _one = train_cnn(build_cnn(CNN_CFG["dropout"]), X_tr_t, y_train, X_val_t, y_val, device,
                       lr=CNN_CFG["lr"], batch_size=CNN_CFG["batch_size"], max_epochs=1, patience=1, seed=SEED)
print("one epoch on %s: %.1f s -> worst case %d epochs ~ %.1f min" % (device, _one["train_seconds"], CNN_CFG["max_epochs"], _one["train_seconds"] * CNN_CFG["max_epochs"] / 60))

In [ ]:
set_global_seed(SEED)                                            # seed BEFORE building: reproducible initial weights
cnn_model = build_cnn(CNN_CFG["dropout"])
cnn_model, history, cnn_info = train_cnn(cnn_model, X_tr_t, y_train, X_val_t, y_val, device,
                                         lr=CNN_CFG["lr"], batch_size=CNN_CFG["batch_size"],
                                         max_epochs=CNN_CFG["max_epochs"], patience=CNN_CFG["patience"],
                                         seed=SEED, checkpoint_path=RESULTS_DIR / "cnn_best.pt")
print({k: v for k, v in cnn_info.items() if k != "epoch_seconds"})
fig = plot_learning_curves(history, cnn_info["best_epoch"]); fig.savefig(FIGURES_DIR / "cnn_learning_curves.png", dpi=150); plt.show()

In [ ]:
# overfit evidence: eval-mode accuracy on the full train split vs validation (restored best weights)
p_tr, _ = predict_cnn(cnn_model, X_tr_t, device); p_va, _ = predict_cnn(cnn_model, X_val_t, device)
cnn_train_eval = evaluate_predictions(y_train, p_tr, "cnn_train"); cnn_val_eval = evaluate_predictions(y_val, p_va, "cnn_val")
print("CNN train acc %.4f | val acc %.4f | gap %.4f" % (cnn_train_eval["accuracy"], cnn_val_eval["accuracy"], cnn_train_eval["accuracy"] - cnn_val_eval["accuracy"]))
assert abs(cnn_val_eval["accuracy"] - cnn_info["best_val_acc"]) < 1e-6, "restored weights do not reproduce best val acc"

## 8. FREEZE, then the single final evaluation on the held-out images

In [ ]:
def _k(key): return "|".join(key) if isinstance(key, tuple) else str(key)
def _plain(o): return json.loads(json.dumps(o, default=lambda x: x.item() if hasattr(x, "item") else str(x)))

selected = {
    "classical_best_params": {_k(k): v for k, v in best_params.items()},
    "classical_fit_info": {_k(k): v for k, v in fit_infos.items()} if isinstance(fit_infos, dict) else fit_infos,
    "cnn": {**CNN_CFG, "best_epoch": cnn_info["best_epoch"], "best_val_loss": cnn_info["best_val_loss"],
            "best_val_acc": cnn_info["best_val_acc"], "train_mean": cnn_mean, "train_std": cnn_std,
            "trainable_params": count_trainable_params(cnn_model)},
    "hog_params": {k: list(v) if isinstance(v, tuple) else v for k, v in HOG_PARAMS.items()},
}
FROZEN_PATH = RESULTS_DIR / "selected_configs.json"
frozen_sha = freeze_selection(_plain(selected), FROZEN_PATH)
print("frozen:", FROZEN_PATH, "sha256:", frozen_sha)

In [ ]:
from src.experiments import run_test_gate
# Section 8 only: load and validate the official held-out test split.
X_test, y_test = load_test_dataset()
print(verify_test_dataset(X_test, y_test))

def cnn_predict_fn(X_uint8):
    # Normalise with TRAIN mean/std, then predict in evaluation mode.
    return predict_cnn(cnn_model, make_cnn_tensor(X_uint8, cnn_mean, cnn_std), device)

gate = run_test_gate(FROZEN_PATH, models, X_test, y_test, cnn_predict_fn=cnn_predict_fn)
np.savez(RESULTS_DIR / "test_gate_outputs.npz", **{_k(k).replace("|", "__") + "__pred": v["y_pred"] for k, v in gate.items()})
print("models evaluated:", [_k(k) for k in gate])


## 9. Final evaluation of all 9 models (bootstrap CI, McNemar)

In [ ]:
from src.evaluation import mcnemar_test
evals, cis = {}, {}
for key, out in gate.items():
    evals[key] = evaluate_predictions(y_test, out["y_pred"], _k(key))
    cis[key] = {m: bootstrap_ci(y_test, out["y_pred"], metric=m, n_boot=1000, seed=SEED) for m in ("accuracy", "macro_f1")}
summary = pd.DataFrame([{"model": _k(k), "accuracy": e["accuracy"], "acc_lo": cis[k]["accuracy"][1], "acc_hi": cis[k]["accuracy"][2],
                         "macro_f1": e["macro_f1"], "f1_lo": cis[k]["macro_f1"][1], "f1_hi": cis[k]["macro_f1"][2]}
                        for k, e in evals.items()]).sort_values("macro_f1", ascending=False)
display(summary.round(4))
# CNN vs each classical model: McNemar (8 comparisons -> Bonferroni-adjusted p reported)
mc = []
for key in gate:
    if key == "cnn": continue
    r = mcnemar_test(y_test, gate["cnn"]["y_pred"], gate[key]["y_pred"])
    mc.append({"vs": _k(key), **r, "p_bonferroni": min(1.0, r["p_value"] * (len(gate) - 1))})
mcnemar_df = pd.DataFrame(mc); display(mcnemar_df)
summary.to_csv(RESULTS_DIR / "metrics_with_ci.csv", index=False)
mcnemar_df.to_csv(RESULTS_DIR / "mcnemar_cnn_vs_classical.csv", index=False)
pd.DataFrame({_k(k): e["per_class_recall"] for k, e in evals.items()}, index=CLASS_NAMES).T.to_csv(RESULTS_DIR / "per_class_recall.csv")
np.savez(RESULTS_DIR / "confusion_matrices.npz", **{_k(k).replace("|", "__"): e["confusion"] for k, e in evals.items()})

In [ ]:
# diagnostic only (reports, alters nothing): exact duplicates between train and held-out images
print(find_duplicates(X_train_full, X_test))

## 10. Inference benchmarks and parameter counts (same images, no shuffling)

In [ ]:
feat_fn = {"pixels": flatten_pixels, "hog": lambda X: extract_hog_features(X, **HOG_PARAMS)}
feats_ho = {rep: cached_features(X_test, rep, "test") for rep in feat_fn}      # read from cache; timing only, labels unused
bench = []
for (name, rep), pipe in models.items():
    for r in (benchmark_classical_inference(pipe, feats_ho[rep], repeats=3),
              benchmark_classical_inference(pipe, X_test, repeats=3, feature_fn=feat_fn[rep])):
        bench.append({"model": name, "rep": rep, **r})
X_ho_t = make_cnn_tensor(X_test, cnn_mean, cnn_std)                              # pre-normalised, outside the timed region
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    r = benchmark_cnn_inference(cnn_model, X_ho_t, torch.device(dev), batch_size=BENCH_BATCH_SIZE, warmup_batches=3, repeats=5)
    bench.append({"model": "cnn", "rep": "tensor", **r})
bench_df = pd.DataFrame(bench)
bench_df.to_json(RESULTS_DIR / "benchmarks.json", orient="records", indent=2)
display(bench_df.drop(columns=["repeats_s"]).round(4))

params = {"cnn": (count_trainable_params(cnn_model), "trainable parameters")}
for key, pipe in models.items(): params[_k(key)] = classical_param_count(pipe)
display(pd.DataFrame([{"model": k, "count": v[0], "note": v[1]} for k, v in params.items()]))

## 11. Error analysis

In [ ]:
cm_cnn = evals["cnn"]["confusion"]
display(top_confusions(cm_cnn, k=10).round(3))
fig = plot_confusion_matrix(cm_cnn, "CNN: confusion matrix (held-out images)"); fig.savefig(FIGURES_DIR / "cm_cnn.png", dpi=150); plt.show()
best_classical = max((k for k in gate if k != "cnn"), key=lambda k: evals[k]["macro_f1"])   # NOTE: chosen post hoc, descriptive only
print("best classical by held-out macro-F1 (descriptive, post hoc):", _k(best_classical))
fig = plot_confusion_matrix(evals[best_classical]["confusion"], _k(best_classical)); plt.show()
display(compare_errors(y_test, {_k(k): v["y_pred"] for k, v in gate.items()}))
for mode in ("correct", "incorrect"):
    fig = show_prediction_examples(X_test, y_test, gate["cnn"]["y_pred"], gate["cnn"]["conf"], mode=mode, n=12, title="CNN " + mode); plt.show()
fig = plot_per_class_recall_heatmap({_k(k): e for k, e in evals.items()}); fig.savefig(FIGURES_DIR / "per_class_recall.png", dpi=150); plt.show()

## 12. Results table, comparison, findings (all values computed)

In [ ]:
NAME = {"logreg": "Logistic regression", "knn": "k-NN", "rf": "Random forest", "svm": "RBF-SVM", "cnn": "Small CNN"}
TEX_NAME = {**NAME, "knn": "$k$-NN"}
REP = {"pixels": "pixels", "hog": "HOG", "tensor": "learned"}

def _ms(model, rep, device, end_to_end):
    sel = bench_df[(bench_df.model == model) & (bench_df.rep == rep) & (bench_df.device == device)
                   & (bench_df.includes_preprocessing == end_to_end)]
    return float(sel.per_image_ms.iloc[0]) if len(sel) else None

entries = []
for key, e in evals.items():
    name, rep = key if isinstance(key, tuple) else ("cnn", "tensor")
    pcount, pnote = params[_k(key)]
    if name == "cnn":
        cfg = "Adam lr=%g, batch %d, dropout %g, best epoch %d (max %d, patience %d)" % (
            CNN_CFG["lr"], CNN_CFG["batch_size"], CNN_CFG["dropout"], cnn_info["best_epoch"], CNN_CFG["max_epochs"], CNN_CFG["patience"])
        n_used, fit_s = N_TRAIN, cnn_info["train_seconds"]
        notes = "single seed %d; fit_seconds includes per-epoch validation passes" % SEED
        po, e2e, gpu = _ms("cnn", "tensor", "cpu", False), "not measured (inputs are pre-normalised tensors)", _ms("cnn", "tensor", "cuda", False)
        gpu = gpu if gpu is not None else "no GPU available"
    else:
        fi = fit_infos[key]
        cfg = ", ".join("%s=%s" % kv for kv in fi["params"].items())
        n_used, fit_s = fi["n_train_used"], fi["fit_seconds"]
        notes = ("hyperparameters tuned on a stratified subset of %d of %d training images" % (min(SVM_TUNE_N, N_TRAIN), N_TRAIN)) if name == "svm" else ""
        if fi.get("gated"):
            notes = (notes + "; " if notes else "") + "final fit on a subset (runtime gate)"
        po, e2e, gpu = _ms(name, rep, "cpu", False), _ms(name, rep, "cpu", True), "n/a (CPU only)"
    entries.append(dict(model=NAME[name], representation=REP[rep], training_config=cfg, n_train_used=n_used,
                        accuracy=e["accuracy"], macro_f1=e["macro_f1"],
                        infer_cpu_predict_only_ms_per_img=po, infer_cpu_end_to_end_ms_per_img=e2e, infer_gpu_ms_per_img=gpu,
                        params=(pcount if pcount is not None else "n/a"), params_note=pnote, fit_seconds=fit_s, notes=(notes or "-")))
results_df = build_results_table(entries)                       # missing values -> "TBD"; unknown keys raise
save_results_table(results_df, RESULTS_DIR / "results_table.csv", RESULTS_DIR / "results_table.md")
display(results_df)
fig = plot_model_comparison(summary); fig.savefig(FIGURES_DIR / "model_comparison.png", dpi=150); plt.show()

In [ ]:
# LaTeX tables for the report, generated from the measured values (never retyped by hand)
def _f(x, nd=4): return "TBD" if x is None else ("%.*f" % (nd, x) if isinstance(x, (int, float)) else str(x))
rows = ["\\begin{table*}[t]", "\\centering",
        "\\caption{Test results for the nine models, measured once at the test gate (95\\% bootstrap confidence intervals, 1000 resamples).}",
        "\\label{tab:results}", "\\begin{tabular}{llcc}", "\\toprule", "Model & Representation & Accuracy [95\\% CI] & Macro-F1 [95\\% CI] \\\\", "\\midrule"]
for key in gate:
    name, rep = key if isinstance(key, tuple) else ("cnn", "tensor")
    a, alo, ahi = cis[key]["accuracy"]; f1, flo, fhi = cis[key]["macro_f1"]
    rows.append("%s & %s & %.4f [%.4f, %.4f] & %.4f [%.4f, %.4f] \\\\" % (TEX_NAME[name], REP[rep], a, alo, ahi, f1, flo, fhi))
rows += ["\\bottomrule", "\\end{tabular}", "\\end{table*}"]
(ROOT / "report").mkdir(exist_ok=True)
(ROOT / "report" / "results_table.tex").write_text("\n".join(rows) + "\n", encoding="utf-8")

hw = "%s; %d logical CPU cores" % ("GPU+CPU" if torch.cuda.is_available() else "CPU only", os.cpu_count())
rows = ["\\begin{table}[t]", "\\centering",
        "\\caption{Inference time per image on the 10{,}000 test images (%s; medians, model loading excluded) and parameter counts. ``Predict only'' uses precomputed features; ``end-to-end'' includes feature extraction from raw images. CNN inputs are pre-normalised tensors. Parameters: trainable weights for the CNN and logistic regression, total tree nodes for the random forest, support vectors for the SVM (not comparable to weights); $k$-NN is non-parametric.}" % hw,
        "\\label{tab:inference}", "\\footnotesize", "\\begin{tabular}{llrrr}", "\\toprule",
        "Model & Repr. & Predict only (ms) & End-to-end (ms) & Params \\\\", "\\midrule"]
for e in entries:
    nm = {v: k for k, v in NAME.items()}[e["model"]]
    p = e["params"]
    rows.append("%s & %s & %s & %s & %s \\\\" % (TEX_NAME[nm], e["representation"], _f(e["infer_cpu_predict_only_ms_per_img"], 3),
                _f(e["infer_cpu_end_to_end_ms_per_img"], 3) if not isinstance(e["infer_cpu_end_to_end_ms_per_img"], str) else "--",
                "n/a" if isinstance(p, str) else "{:,}".format(int(p)).replace(",", "{,}")))
rows += ["\\bottomrule", "\\end{tabular}", "\\end{table}"]
(ROOT / "report" / "inference_table.tex").write_text("\n".join(rows) + "\n", encoding="utf-8")
print("wrote report/results_table.tex and report/inference_table.tex")

In [ ]:
top = summary.iloc[0]
cnn_rank = list(summary.model).index("cnn") + 1
print("Measured: highest test macro-F1 = %s (%.4f). CNN rank by macro-F1: %d of %d (macro-F1 %.4f)." % (top.model, top.macro_f1, cnn_rank, len(summary), float(summary[summary.model == "cnn"].macro_f1.iloc[0])))
print("Interpret together with the CIs and McNemar table above; differences inside overlapping CIs are not evidence of a ranking.")

## 13. Limitations and reproducibility notes
* Single run (seed 42) for the headline CNN; run-to-run variance across seeds is **not** measured here (stretch experiment, reported separately if done).
* Models are trained on the 54k split only (no refit on train+val); any subsampling is printed in section 6 (`n_train_used`) and stored in `results/fit_info.json`.
* Fashion-MNIST is 28x28 grayscale studio-style images; conclusions do not transfer automatically to real inventory photos.
* Timings depend on the hardware recorded in section 0 / `results/benchmarks.json`; classical "predict only" includes the in-pipeline scaler; the GPU row exists only if a GPU was present.
* The held-out evaluation can run only once per `results/` folder (flag file); to re-run from scratch delete `results/test_gate_done.flag` deliberately and document why.
* Best-classical selection in section 11 is post hoc and descriptive only; model selection was frozen in section 8 before any held-out use.

In [ ]:
print(json.dumps(env, indent=2, default=str))
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True); print(r.stdout[-1500:])